In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

dataset_path = kagglehub.dataset_download("shriyashjagtap/fraudulent-e-commerce-transactions")
df = pd.read_csv(dataset_path+"/Fraudulent_E-Commerce_Transaction_Data_2.csv")

/home/arav/anaconda3/envs/razor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def clean_data(df) -> pd.DataFrame:
    df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])
    
    df['Transaction Day'] = df["Transaction Date"].dt.day
    df["Transaction DOW"] = df["Transaction Date"].dt.day_of_week
    df["Transaction Month"] = df["Transaction Date"].dt.month
    
    ''' 
    The lower fence of the customer age is 9. We will replace values between -9 and 8 with the mean, 
    and values less than -9 will be replaced with their absolute values.
    '''
    mean_value = np.round(df['Customer Age'].mean(),0) 
    df['Customer Age'] = np.where(df['Customer Age'] <= -9, 
                                    np.abs(df['Customer Age']), 
                                    df['Customer Age'])

    df['Customer Age'] = np.where(df['Customer Age'] < 9, 
                                    mean_value, 
                                    df['Customer Age'])
    
    df["Is Address Match"] = (df["Shipping Address"] == df["Billing Address"]).astype(int)
    
    df.drop(columns=["Transaction ID", "Customer ID", "Customer Location",
                     "IP Address", "Transaction Date","Shipping Address","Billing Address"], inplace=True)
    
    int_col = df.select_dtypes(include="int").columns
    float_col = df.select_dtypes(include="float").columns
    
    df[int_col] = df[int_col].apply(pd.to_numeric, downcast='integer')
    df[float_col] = df[float_col].apply(pd.to_numeric, downcast='float')
    
    return df

In [3]:
df = clean_data(df)
X = df.drop(columns=["Is Fraudulent"])
y = df["Is Fraudulent"]

In [4]:
cat_col = X.select_dtypes(include="O").columns
num_col = []
for col in X.columns:
    if col not in cat_col  and col != 'Is Address Match':
        num_col.append(col)

In [5]:
transformer = ColumnTransformer(transformers=[
    ('encoding',OneHotEncoder(),cat_col),
    ('scaling',StandardScaler(),num_col)
],remainder='passthrough')

In [ ]:
import optuna
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# 1. Stratified split ensures the 5% fraud ratio is maintained in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Calculate dynamic class weight for the training set
num_neg = np.sum(y_train == 0)
num_pos = np.sum(y_train == 1)
scale_pos_weight = num_neg / num_pos if num_pos > 0 else 1.0

def objective(trial):
    params = {
        "tree_method": "hist",
        "device": "cuda",
        "scale_pos_weight": scale_pos_weight,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 1500), 
        "max_depth": trial.suggest_int("max_depth", 1, 8), 
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
    }
    
    classifier = XGBClassifier(**params, random_state=42)
    
    model = Pipeline(steps=[
        ('transformer', transformer),
        ('classifier', classifier)
    ])
    
    model.fit(X_train, y_train)

    y_probs = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, y_probs)

    return pr_auc

study = optuna.create_study(direction="maximize")
# Increased trials since it only takes ~1 minute per 70 trials
study.optimize(objective, n_trials=150)

print("Best Hyperparameters:", study.best_params)
print("Best PR-AUC:", study.best_trial.value)

[I 2026-08-30 23:46:40,679] A new study created in memory with name: no-name-07a3cc6f-2df1-4724-b315-8347b84c27ce


[I 2026-08-30 23:46:41,288] Trial 0 finished with value: 0.39634769758082905 and parameters: {'learning_rate': 0.02155928331551792, 'n_estimators': 1273, 'max_depth': 1, 'min_child_weight': 18, 'gamma': 2.3913459384362867, 'subsample': 0.6286390167476206, 'colsample_bytree': 0.9914754011632512, 'reg_alpha': 4.640803878838705}. Best is trial 0 with value: 0.39634769758082905.
[I 2026-08-30 23:46:41,616] Trial 1 finished with value: 0.38203002464136004 and parameters: {'learning_rate': 0.009772294764879088, 'n_estimators': 257, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1.1570069737457067, 'subsample': 0.6298750709326282, 'colsample_bytree': 0.9977631970701231, 'reg_alpha': 1.1118036818797687}. Best is trial 0 with value: 0.39634769758082905.
[I 2026-08-30 23:46:42,362] Trial 2 finished with value: 0.3494701801731746 and parameters: {'learning_rate': 0.08750585535345597, 'n_estimators': 1490, 'max_depth': 5, 'min_child_weight': 15, 'gamma': 4.647764305888778, 'subsample': 0.88992718

Best Hyperparameters: {'learning_rate': 0.005187544455303851, 'n_estimators': 1028, 'max_depth': 2, 'min_child_weight': 10, 'gamma': 1.5116687959058668, 'subsample': 0.8180714267188646, 'colsample_bytree': 0.9926524228665097, 'reg_alpha': 0.22366552184703914}
Best PR-AUC: 0.41497152877638527
